In [1]:
import os
import re
from tifffile import imread, imwrite
import numpy as np
data_directory = r"E:\Data\2025_04_22 60xWIA 1.5x CRISPRi mCh\H12\sharpest_slice"
file_list = os.listdir(data_directory)
print("Number of files found:", len(file_list))
print("First 10 files:", file_list[:10])  # Print a preview of the first 10 files

Number of files found: 96
First 10 files: ['WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_analysis.csv', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_analysis.png', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_Fluoro.csv', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_Fluoro_Slice_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_intensity_profile.png', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_sharpest_image_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_analysis.csv', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_analysis.png', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_Fluoro.csv', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_Fluoro_Slice_5.tif']


In [2]:
# === EXTRACTING FILENAME COMPONENTS ===
def extract_filename_components(dic_fname, fluor_fname):
    """
    Extracts Well, Seq, Z from the DIC filename and slice number from the Fluoro filename.
    Returns a meaningful stack filename.
    """
    match_dic = re.search(r"(Well[A-H]\d{2}).*?(Seq\d+_Z\d+)", dic_fname)
    match_fluor = re.search(r"Slice_(\d+)", fluor_fname)
    if match_dic and match_fluor:
        well, seq_z = match_dic.groups()
        slice_num = match_fluor.group(1)
        return f"{well}_{seq_z}_slice{slice_num}_stack"
    else:
        return None  # fallback to default name if parsing fails

In [3]:
def create_data_directory(data_directory):
    all_file_names = [f.strip() for f in os.listdir(data_directory) if f.endswith('.tif')]
    dic_files = [f for f in all_file_names if 'sharpest_image' in f]
    non_dic_files = [f for f in all_file_names if 'Fluoro_Slice' in f]

    print(f"Total DIC files: {len(dic_files)}")
    print(f"Total Fluoro files: {len(non_dic_files)}")

    paired_files_dict = {}
    for index, (dic_file, fluor_file) in enumerate(zip(dic_files, non_dic_files)):
        paired_files_dict[index] = (dic_file, fluor_file)

    print(f"Paired files: {len(paired_files_dict)} matched pairs")
    if len(dic_files) != len(non_dic_files):
        print("Warning: The number of DIC and Fluoro files does not match.")
    
    return paired_files_dict

# Call the function
paired_files_dict = create_data_directory(data_directory)
# Optionally, print the first 10 pairs for debugging
print("First 10 pairs:", {key: value for key, value in list(paired_files_dict.items())[:10]})

Total DIC files: 16
Total Fluoro files: 16
Paired files: 16 matched pairs
First 10 pairs: {0: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_sharpest_image_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_Fluoro_Slice_4.tif'), 1: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_sharpest_image_5.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_Fluoro_Slice_5.tif'), 2: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z12_sharpest_image_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z12_Fluoro_Slice_4.tif'), 3: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z13_sharpest_image_6.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z13_Fluoro_Slice_6.tif'), 4: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z14_sharpest_image_6.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z14_Fluoro_Slice_6.tif'), 5: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z15_sharpest_image_6.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z15_Fluoro_Slice_6.tif'), 6: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z16_

In [4]:
# === STACK AND SAVE IMAGES ===
def combine_to_stack(data_directory, paired_files_dict, output_subfolder="stacked"):
    output_dir = os.path.join(data_directory, output_subfolder)
    os.makedirs(output_dir, exist_ok=True)
    
    for idx, (dic_file, fluor_file) in paired_files_dict.items():
        dic_path = os.path.join(data_directory, dic_file)
        fluor_path = os.path.join(data_directory, fluor_file)

        try:
            dic_img = imread(dic_path)
            fluor_img = imread(fluor_path)
        except Exception as e:
            print(f"Error reading files {dic_file} or {fluor_file}: {e}")
            continue

        if dic_img.shape != fluor_img.shape:
            print(f"Shape mismatch for {dic_file} and {fluor_file}, skipping...")
            continue

        stacked_img = np.stack([dic_img, fluor_img], axis=0)

        output_basename = extract_filename_components(dic_file, fluor_file)
        if output_basename is None:
            output_basename = f"stacked_{idx:04d}"
        output_filename = f"{output_basename}.tif"
        output_path = os.path.join(output_dir, output_filename)

        imwrite(output_path, stacked_img)
        print(f"Saved {output_path}")

In [5]:
# === RUN SCRIPT ===
if __name__ == "__main__":
    print("Scanning folder:", data_directory)
    file_list = os.listdir(data_directory)
    print("Number of files found:", len(file_list))
    paired_files_dict = create_data_directory(data_directory)
    print("First 10 pairs:", {k: v for k, v in list(paired_files_dict.items())[:10]})

    combine_to_stack(data_directory, paired_files_dict)

Scanning folder: E:\Data\2025_04_22 60xWIA 1.5x CRISPRi mCh\H12\sharpest_slice
Number of files found: 96
Total DIC files: 16
Total Fluoro files: 16
Paired files: 16 matched pairs
First 10 pairs: {0: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_sharpest_image_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z10_Fluoro_Slice_4.tif'), 1: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_sharpest_image_5.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z11_Fluoro_Slice_5.tif'), 2: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z12_sharpest_image_4.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z12_Fluoro_Slice_4.tif'), 3: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z13_sharpest_image_6.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z13_Fluoro_Slice_6.tif'), 4: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z14_sharpest_image_6.tif', 'WellH12_ChannelCam-DIA DIC Master_Seq0168_Z14_Fluoro_Slice_6.tif'), 5: ('WellH12_ChannelCam-DIA DIC Master_Seq0168_Z15_sharpest_image_6.tif', 'WellH12_Channe